# Mini Project 1

We aim to construct a low risk portfolio as measured by the variance of daily returns, and compare the returns of such a portfolio to one that simply chases returns, i.e. a potentially risky portfolio.   

In [6]:
import pandas as pd
import yfinance as yf
import json
import datetime
import os
import time
import pytz
import numpy as np
import cvxpy as cp
import mp1mp2_funcs

In [40]:
# Code to get symbols of S&P 500 companies.
# symbols = pd.read_html(
#     "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
# )[0]["Symbol"].tolist()

In [4]:
# Symbols of S&P 500 companies as in May 2025
symbols = ['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'ANSS',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APTV',
 'ACGL',
 'ADM',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVB',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK.B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'BK',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF.B',
 'BLDR',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CZR',
 'CPT',
 'CPB',
 'COF',
 'CAH',
 'KMX',
 'CCL',
 'CARR',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'CAG',
 'COP',
 'ED',
 'STZ',
 'CEG',
 'COO',
 'CPRT',
 'GLW',
 'CPAY',
 'CTVA',
 'CSGP',
 'COST',
 'CTRA',
 'CRWD',
 'CCI',
 'CSX',
 'CMI',
 'CVS',
 'DHR',
 'DRI',
 'DVA',
 'DAY',
 'DECK',
 'DE',
 'DELL',
 'DAL',
 'DVN',
 'DXCM',
 'FANG',
 'DLR',
 'DG',
 'DLTR',
 'D',
 'DPZ',
 'DASH',
 'DOV',
 'DOW',
 'DHI',
 'DTE',
 'DUK',
 'DD',
 'EMN',
 'ETN',
 'EBAY',
 'ECL',
 'EIX',
 'EW',
 'EA',
 'ELV',
 'EMR',
 'ENPH',
 'ETR',
 'EOG',
 'EPAM',
 'EQT',
 'EFX',
 'EQIX',
 'EQR',
 'ERIE',
 'ESS',
 'EL',
 'EG',
 'EVRG',
 'ES',
 'EXC',
 'EXE',
 'EXPE',
 'EXPD',
 'EXR',
 'XOM',
 'FFIV',
 'FDS',
 'FICO',
 'FAST',
 'FRT',
 'FDX',
 'FIS',
 'FITB',
 'FSLR',
 'FE',
 'FI',
 'F',
 'FTNT',
 'FTV',
 'FOXA',
 'FOX',
 'BEN',
 'FCX',
 'GRMN',
 'IT',
 'GE',
 'GEHC',
 'GEV',
 'GEN',
 'GNRC',
 'GD',
 'GIS',
 'GM',
 'GPC',
 'GILD',
 'GPN',
 'GL',
 'GDDY',
 'GS',
 'HAL',
 'HIG',
 'HAS',
 'HCA',
 'DOC',
 'HSIC',
 'HSY',
 'HES',
 'HPE',
 'HLT',
 'HOLX',
 'HD',
 'HON',
 'HRL',
 'HST',
 'HWM',
 'HPQ',
 'HUBB',
 'HUM',
 'HBAN',
 'HII',
 'IBM',
 'IEX',
 'IDXX',
 'ITW',
 'INCY',
 'IR',
 'PODD',
 'INTC',
 'ICE',
 'IFF',
 'IP',
 'IPG',
 'INTU',
 'ISRG',
 'IVZ',
 'INVH',
 'IQV',
 'IRM',
 'JBHT',
 'JBL',
 'JKHY',
 'J',
 'JNJ',
 'JCI',
 'JPM',
 'JNPR',
 'K',
 'KVUE',
 'KDP',
 'KEY',
 'KEYS',
 'KMB',
 'KIM',
 'KMI',
 'KKR',
 'KLAC',
 'KHC',
 'KR',
 'LHX',
 'LH',
 'LRCX',
 'LW',
 'LVS',
 'LDOS',
 'LEN',
 'LII',
 'LLY',
 'LIN',
 'LYV',
 'LKQ',
 'LMT',
 'L',
 'LOW',
 'LULU',
 'LYB',
 'MTB',
 'MPC',
 'MKTX',
 'MAR',
 'MMC',
 'MLM',
 'MAS',
 'MA',
 'MTCH',
 'MKC',
 'MCD',
 'MCK',
 'MDT',
 'MRK',
 'META',
 'MET',
 'MTD',
 'MGM',
 'MCHP',
 'MU',
 'MSFT',
 'MAA',
 'MRNA',
 'MHK',
 'MOH',
 'TAP',
 'MDLZ',
 'MPWR',
 'MNST',
 'MCO',
 'MS',
 'MOS',
 'MSI',
 'MSCI',
 'NDAQ',
 'NTAP',
 'NFLX',
 'NEM',
 'NWSA',
 'NWS',
 'NEE',
 'NKE',
 'NI',
 'NDSN',
 'NSC',
 'NTRS',
 'NOC',
 'NCLH',
 'NRG',
 'NUE',
 'NVDA',
 'NVR',
 'NXPI',
 'ORLY',
 'OXY',
 'ODFL',
 'OMC',
 'ON',
 'OKE',
 'ORCL',
 'OTIS',
 'PCAR',
 'PKG',
 'PLTR',
 'PANW',
 'PARA',
 'PH',
 'PAYX',
 'PAYC',
 'PYPL',
 'PNR',
 'PEP',
 'PFE',
 'PCG',
 'PM',
 'PSX',
 'PNW',
 'PNC',
 'POOL',
 'PPG',
 'PPL',
 'PFG',
 'PG',
 'PGR',
 'PLD',
 'PRU',
 'PEG',
 'PTC',
 'PSA',
 'PHM',
 'PWR',
 'QCOM',
 'DGX',
 'RL',
 'RJF',
 'RTX',
 'O',
 'REG',
 'REGN',
 'RF',
 'RSG',
 'RMD',
 'RVTY',
 'ROK',
 'ROL',
 'ROP',
 'ROST',
 'RCL',
 'SPGI',
 'CRM',
 'SBAC',
 'SLB',
 'STX',
 'SRE',
 'NOW',
 'SHW',
 'SPG',
 'SWKS',
 'SJM',
 'SW',
 'SNA',
 'SOLV',
 'SO',
 'LUV',
 'SWK',
 'SBUX',
 'STT',
 'STLD',
 'STE',
 'SYK',
 'SMCI',
 'SYF',
 'SNPS',
 'SYY',
 'TMUS',
 'TROW',
 'TTWO',
 'TPR',
 'TRGP',
 'TGT',
 'TEL',
 'TDY',
 'TER',
 'TSLA',
 'TXN',
 'TPL',
 'TXT',
 'TMO',
 'TJX',
 'TKO',
 'TSCO',
 'TT',
 'TDG',
 'TRV',
 'TRMB',
 'TFC',
 'TYL',
 'TSN',
 'USB',
 'UBER',
 'UDR',
 'ULTA',
 'UNP',
 'UAL',
 'UPS',
 'URI',
 'UNH',
 'UHS',
 'VLO',
 'VTR',
 'VLTO',
 'VRSN',
 'VRSK',
 'VZ',
 'VRTX',
 'VTRS',
 'VICI',
 'V',
 'VST',
 'VMC',
 'WRB',
 'GWW',
 'WAB',
 'WBA',
 'WMT',
 'DIS',
 'WBD',
 'WM',
 'WAT',
 'WEC',
 'WFC',
 'WELL',
 'WST',
 'WDC',
 'WY',
 'WSM',
 'WMB',
 'WTW',
 'WDAY',
 'WYNN',
 'XEL',
 'XYL',
 'YUM',
 'ZBRA',
 'ZBH',
 'ZTS']

We first pick stocks with high returns. Given a 'end_year' we look for stocks with the highest returns over the previous 1,2,3 and 4 years averaged over the past one year. For each time horizon, we pick 5 stocks giving us a total of 20 stocks. A portfolio with all these stocks with equal weights is our candidate for a high return portfolio. We do not aim to just find a high risk portfolio, since there is no point of simply chasing risk. We look for returns and accept whatever risk that comes with it. 

In [ ]:
def get_high_ret_stocks(symbols, end_year):
    """Identifies top-performing stocks based on historical returns over multiple timeframes.

    This function calculates the return for a given list of stock symbols over several time horizons 
    (1, 2, 3, and 4 years) averaged over 1 year. For each time horizon, it
    selects the top 5 stocks with the highest average returns.

    The selection process is cumulative. If a stock is picked as a top performer for one
    timeframe, it will not be picked again for another, ensuring a diverse set of
    high-performing stocks is returned.

    Args:
        symbols (list of str): A list of stock ticker symbols to be analyzed.
        end_year (int): The end year for the analysis period. The calculation of
                        average returns will be based on the year leading up to
                        May 1st of this `end_year`.

    Returns:
        set: A set of unique stock symbols that were identified as top performers
             across the different time horizons evaluated.
    """
    selected_symbols_highrets = set()
    eastern = pytz.timezone('US/Eastern')

    for numyears in range(1,5):
        avg_returns = []
        for symbol in symbols:
            stockprices = pd.read_csv(f'StockPriceData/{symbol}.csv')
            stockprices['Date'] = pd.to_datetime(stockprices['Date'], utc=True)
            stockprices['Date'] = stockprices['Date'].dt.tz_convert('US/Eastern')
            stockprices = stockprices[stockprices['Date']>=eastern.localize(datetime.datetime(end_year-6, 5, 1, 0, 0, 0))]
            stockprices['close_prev'] = stockprices['Close'].shift(252*numyears)
            stockprices['growth'] = stockprices['Close']/stockprices['close_prev']
            avg_returns.append((symbol,stockprices[(stockprices['Date']>=eastern.localize(datetime.datetime(end_year-1, 5, 1, 0, 0, 0)))&(stockprices['Date']<eastern.localize(datetime.datetime(end_year, 5, 1, 0, 0, 0)))]['growth'].mean()))   
        df = pd.DataFrame(avg_returns, columns=['Symbol', 'Avg_return'])
        df.sort_values(by='Avg_return', ascending=False, inplace=True)
        picked = 0
        i=0
        while picked<5:
            if df.iloc[i]['Symbol'] not in selected_symbols_highrets:
                selected_symbols_highrets.add(df.iloc[i]['Symbol'])
                picked+=1
            i+=1

    return list(selected_symbols_highrets)

In [ ]:

len(get_high_ret_stocks(symbols, 2025))

20

# Low Risk

Diversification across sectors is a good way to reduce risk, since factors influencing stock price movements impact companies across different sectors differently. So we select candidates for our low risk portfolio from a set of the top 5 performing stocks in each of the 11 GICS sectors.

In [11]:
def get_top_stocks_by_sector(end_year):
    """Selects the top 5 performing stocks from each S&P 500 sector.

    This function scrapes the list of S&P 500 companies and their respective
    GICS sectors from Wikipedia. For each sector, it calculates the average
    one-year return for every stock within that sector. The return is evaluated
    for the period between May 1st of the year prior to `end_year` and May 1st
    of `end_year`.

    The function then identifies the 5 stocks with the highest average returns
    from each sector and compiles them into a single set.

    Args:
        end_year (int): The terminal year for the analysis. The performance
                        of stocks is measured over the 1-year period ending
                        on May 1st of this year.

    Returns:
        set: A set of stock symbols representing the top 5 performers from
             each GICS sector for the specified one-year period.
    """
    selected_symbols = set()
    eastern = pytz.timezone('US/Eastern')
    sp500table = pd.read_html(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    )
    grouped_dfs = {group: sub_df for group, sub_df in sp500table[0].groupby(by='GICS Sector')}

    for key in grouped_dfs.keys():
        avg_returns = []
        for symbol in grouped_dfs[key]['Symbol'].to_list():
            stockprices = pd.read_csv(f'StockPriceData/{symbol}.csv')
            stockprices['Date'] = pd.to_datetime(stockprices['Date'], utc=True)
            stockprices['Date'] = stockprices['Date'].dt.tz_convert('US/Eastern')
            stockprices = stockprices[stockprices['Date']>=eastern.localize(datetime.datetime(end_year-4, 5, 1, 0, 0, 0))]
            stockprices['close_prev'] = stockprices['Close'].shift(252)
            stockprices['growth'] = stockprices['Close']/stockprices['close_prev']
            avg_returns.append((symbol,stockprices[(stockprices['Date']>=eastern.localize(datetime.datetime(end_year-1, 5, 1, 0, 0, 0)))&(stockprices['Date']<eastern.localize(datetime.datetime(end_year, 5, 1, 0, 0, 0)))]['growth'].mean()))   
        df = pd.DataFrame(avg_returns, columns=['Symbol', 'Avg_return'])
        df.sort_values(by='Avg_return', ascending=False, inplace=True)
        i = 0
        while i<5:
            selected_symbols.add(df.iloc[i]['Symbol'])
            i+=1

    return selected_symbols

In [12]:
candidates_lowrisk = get_top_stocks_by_sector(2025)

We use a forward selection approach to build the portfolio. That is, we first pick the stock with the smallest variance in daily returns. We then add the stock which gives the smallest variance for our 2 stock portfolio and so on.

In [9]:
def build_low_risk_portfolio(candidates_lowrisk, end_year, num_stocks):
    start_date = datetime.datetime(end_year-1, 5, 1, 0, 0, 0)
    end_date = datetime.datetime(end_year, 5, 1, 0, 0, 0)

    # Get daily returns for our selected candidates and compute the covariances of daily returns
    div_categories_stock_data = mp1mp2_funcs.get_daily_rets(candidates_lowrisk, start_date, end_date)
    div_categories_stock_data.drop(columns=['Date'], inplace=True)
    stock_covs = div_categories_stock_data.cov()

    selected_lowrisk = []
    candidates_lowrisk_copy = candidates_lowrisk.copy()

    while len(selected_lowrisk)<num_stocks:
        cands_for_add = []
        for item in candidates_lowrisk_copy: 
            cands_for_add.append((item, mp1mp2_funcs.find_min_vol_wts(selected_lowrisk+[item], stock_covs)[0]))
        df = pd.DataFrame(cands_for_add, columns=['Symbol', 'Min_vol'])
        df.sort_values(by='Min_vol', ascending=True, inplace=True)
        selected_lowrisk.append(df['Symbol'].iloc[0])
        #print(stock_covs.loc[selected_lowrisk, selected_lowrisk])
        #print(f"Minimum variance with {selected_lowrisk} = {df['Min_vol'].iloc[0]}")
        candidates_lowrisk_copy.discard(df['Symbol'].iloc[0])
    
    variance, weights = mp1mp2_funcs.find_min_vol_wts(selected_lowrisk, stock_covs)
    
    return (selected_lowrisk,weights, variance)
    

In [61]:
selected_lowrisk = build_low_risk_portfolio(candidates_lowrisk, 2025, 20)
selected_lowrisk

(['MO',
  'K',
  'COST',
  'PEG',
  'ECL',
  'NVDA',
  'T',
  'DECK',
  'WELL',
  'PM',
  'SHW',
  'NFLX',
  'DVA',
  'BSX',
  'KMI',
  'AVGO',
  'IFF',
  'TKO',
  'WSM',
  'META'],
 array([0.27089148, 0.25195265, 0.06660221, 0.05776345, 0.01722765,
        0.02354271, 0.07231057, 0.02161504, 0.03221609, 0.04543913,
        0.03213004, 0.01873113, 0.02034197, 0.01905649, 0.02119958,
        0.00725175, 0.01081887, 0.00648045, 0.00198517, 0.00244357]),
 np.float64(4.708523109516651e-05))

# Comparison of risk and returns

In [7]:
high_rets_2025 = (get_high_ret_stocks(symbols, 2025), np.full(20, 0.05, dtype=float))

In [13]:
low_risk_2025 = build_low_risk_portfolio(candidates_lowrisk, 2025, 20)

In [ ]:
mp1mp2_funcs.get_risk_return(high_rets_2025[0], high_rets_2025[1], 2025)
mp1mp2_funcs.get_risk_return(low_risk_2025[0], low_risk_2025[1], 2025)

Average yearly return = 1.9762128518406883, Variance of daily returns = 0.0005282470810472946
Average yearly return = 1.4338199105658718, Variance of daily returns = 4.7085231095166495e-05


We see that we managed to reduce variance of daily returns by a factor of 10 while only reducing annual returns by a factor of 2.

In [ ]:
high_rets_2024 = (get_high_ret_stocks(symbols, 2024), np.full(20, 0.05, dtype=float))
low_risk_2024 = build_low_risk_portfolio(get_top_stocks_by_sector(2024), 2024, 20)
mp1mp2_funcs.get_risk_return(high_rets_2024[0], high_rets_2024[1], 2024)
mp1mp2_funcs.get_risk_return(low_risk_2024[0], low_risk_2024[1], 2024)

Average yearly return = 1.6889331869124315, Variance of daily returns = 0.00015376938368292546
Average yearly return = 1.3121176454517127, Variance of daily returns = 2.7414673261249254e-05
